# EDA 2 — Analyse exploratoire des *features*

### Projet « Terre, Vent, Feu, Eau, Data » · jour 4

---

## Pourquoi une deuxième EDA

L'EDA du jour 2 (`eda_jour2.ipynb`) regardait les **données brutes** : les
incendies tels que la BDIFF les publie. C'était une réalité qu'on découvrait.

Celle-ci regarde les **variables construites** — le tableau que le modèle
mange. Ce n'est plus une réalité qu'on découvre, c'est une **fabrication qu'on
vérifie**. Et une variable fabriquée peut être fausse d'une manière dont une
donnée brute ne peut pas l'être :

| Défaut possible | À quoi ça ressemble |
|---|---|
| **constante** | un calcul raté : tout le monde à zéro |
| **dupliquée** | deux variables à 0,99 de corrélation = une seule information |
| **vide au début** | un historique à 10 ans n'existe pas pour 2006 |
| **trop belle** | elle prédit trop bien… parce qu'elle a vu le futur |

> **Le dernier point est le plus important de ce notebook.** Une variable qui
> prédit anormalement bien n'est presque jamais une découverte. C'est une fuite
> de données.

## Les 8 sections

1. Reconstruction du tableau de variables
2. Inventaire : dimensions, types, valeurs manquantes, colonnes constantes
3. Distributions
4. Le déséquilibre de la cible
5. Corrélations entre variables — chasse aux redondances
6. **Corrélations avec la cible — chasse à la fuite**
7. Les coudes : combien de clusters, et quel rayon
8. Ce que les clusters ont trouvé

Puis une **conclusion** : ce qu'on garde, ce qu'on jette, ce qui reste à faire.

---

⚠️ **Prérequis :** Docker démarré et la base remplie. Ce notebook doit
s'exécuter de bout en bout depuis un noyau vide (*Kernel → Restart & Run All*).

## 0 · Mise en place

On importe **les fonctions de `jour4_model.py`** plutôt que de recopier le
calcul des variables.

**Pourquoi c'est important.** Si on reconstruisait les variables à la main dans
le notebook, on analyserait un tableau *ressemblant* à celui du modèle, mais pas
identique. Le moindre écart et l'EDA validerait quelque chose qui n'est pas ce
qui est entraîné. En important, on a la **garantie** d'étudier exactement le
tableau du modèle.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# racine du depot : le notebook est dans analysis/
RACINE = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(RACINE))

from data.ingestion_pipeline import jour4_model as jm

print("Racine du projet :", RACINE)
print("Periode modelisee :", jm.ANNEE_DEBUT, "->", jm.ANNEE_FIN)
print("Annee de test     :", jm.ANNEE_TEST)
print("Nombre de features:", len(jm.FEATURE_COLUMNS))

## 1 · Reconstruction du tableau de variables

Trois étapes, les mêmes que le modèle :

1. `charger_donnees()` — lit `fires` et `ref_communes` dans PostgreSQL ;
2. `construire_features()` — fabrique le panneau **commune × année** avec
   l'historique décalé et la cible ;
3. `ajouter_clusters()` — ajoute `cluster_risque` (KMeans) et `cluster_spatial`
   (DBSCAN), et nous renvoie au passage les courbes du coude.

Compter une à deux minutes.

In [ ]:
incendies, communes = jm.charger_donnees()
print(f"Incendies charges : {len(incendies):,}")
print(f"Communes chargees : {len(communes):,}")

features_sans_clusters = jm.construire_features(incendies, communes)
features, infos_clusters = jm.ajouter_clusters(features_sans_clusters, incendies)

print(f"\nPanneau construit : {len(features):,} lignes x {features.shape[1]} colonnes")
print(f"Soit {features['code_insee'].nunique():,} communes "
      f"x {features['annee'].nunique()} annees")

### La forme du tableau, et sa limite

Une ligne = **une commune × une année**. La cible répond à :
*« cette commune aura-t-elle au moins un feu l'année suivante ? »*

> ⚠️ **Limite assumée, à documenter dans le rapport.** Le tableau est **annuel**,
> pas mensuel. Les variables `mois_reference`, `mois_sin` et `mois_cos` encodent
> **le mois où la commune brûle habituellement** — sa signature saisonnière —
> et non un mois que l'utilisateur choisirait. Conséquence directe : une commune
> a aujourd'hui le même score en août et en février.
>
> Le sujet demande p. 7 « sélection commune **+ période** ». C'est le principal
> écart qui reste.

In [ ]:
apercu = features.sample(5, random_state=0)[
    ["code_insee", "annee", "nb_feux_5a", "surface_5a_ha",
     "mois_reference", "cluster_risque", "cluster_spatial",
     "cible_incendie_suivant"]
]
apercu

## 2 · Inventaire

Trois questions de base avant toute analyse :

- combien de valeurs manquantes, et **où** ;
- y a-t-il une colonne **constante** (signe d'un calcul raté) ;
- les types sont-ils cohérents.

In [ ]:
inventaire = pd.DataFrame({
    "type": features[jm.FEATURE_COLUMNS].dtypes.astype(str),
    "manquants": features[jm.FEATURE_COLUMNS].isna().sum(),
    "taux_manquants": (features[jm.FEATURE_COLUMNS].isna().mean() * 100).round(2),
    "valeurs_distinctes": features[jm.FEATURE_COLUMNS].nunique(),
    "min": features[jm.FEATURE_COLUMNS].min(numeric_only=True).round(2),
    "max": features[jm.FEATURE_COLUMNS].max(numeric_only=True).round(2),
})
inventaire

In [ ]:
constantes = inventaire[inventaire["valeurs_distinctes"] <= 1].index.tolist()
if constantes:
    print("ALERTE - colonnes constantes (a supprimer) :", constantes)
else:
    print("OK : aucune colonne constante.")

problematiques = inventaire[inventaire["taux_manquants"] > 5].index.tolist()
if problematiques:
    print("\nColonnes a plus de 5 % de manquants :", problematiques)
else:
    print("OK : aucune colonne au-dessus de 5 % de valeurs manquantes.")

### Les valeurs manquantes des premières années

Un historique à 10 ans ne peut pas exister en 2006 : il n'y a pas 10 années
avant. Le code utilise `min_periods=1`, donc la valeur existe mais elle est
**calculée sur moins d'années** — elle n'est pas manquante, elle est
**incomplète**, ce qui est plus sournois.

Regardons la moyenne de `nb_feux_10a` par année : elle doit **monter** au début,
puis se stabiliser. C'est le signe que l'historique se remplit.

In [ ]:
profondeur = features.groupby("annee")[["nb_feux_5a", "nb_feux_10a"]].mean()
ax = profondeur.plot(marker="o")
ax.axvline(2011, color="crimson", ls="--", lw=1)
ax.text(2011.15, ax.get_ylim()[1]*0.9, "debut de l'entrainement",
        color="crimson", fontsize=9)
ax.set_title("Profondeur reelle de l'historique, par annee")
ax.set_ylabel("nombre moyen de feux dans la fenetre")
plt.show()

profondeur.round(3).head(10)

**Ce que ça justifie.** L'entraînement démarre en **2011** et non en 2006 :
avant 2011, la fenêtre à 5 ans n'est pas pleine et celle à 10 ans encore moins.
Entraîner sur ces années apprendrait au modèle que « peu d'historique = peu de
risque », ce qui est un artefact du calcul, pas une vérité du terrain.

C'est une décision de méthode : **à écrire dans le rapport**, pas à laisser
deviner.

## 3 · Distributions

On regarde la forme de chaque variable. Ce qu'on cherche :

- des distributions **très asymétriques** (beaucoup de zéros, quelques valeurs
  énormes) — typiques des surfaces brûlées ;
- des **valeurs aberrantes** ;
- des variables dont l'échelle écrase les autres.

In [ ]:
numeriques = [c for c in jm.FEATURE_COLUMNS
              if pd.api.types.is_numeric_dtype(features[c])]

lignes = int(np.ceil(len(numeriques) / 3))
fig, axes = plt.subplots(lignes, 3, figsize=(15, 3.2 * lignes))
for ax, colonne in zip(axes.ravel(), numeriques):
    valeurs = features[colonne].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(valeurs, bins=40, color="#4C72B0")
    ax.set_title(colonne, fontsize=10)
    ax.set_yscale("log")          # echelle log : sinon on ne voit que la 1re barre
for ax in axes.ravel()[len(numeriques):]:
    ax.axis("off")
fig.suptitle("Distribution des variables (axe vertical en echelle log)", y=1.001)
plt.tight_layout()
plt.show()

**Pourquoi l'axe vertical est en échelle logarithmique.** Sans ça, la première
barre (les communes à zéro feu) écrase tout et les autres sont invisibles. Le
log permet de voir la queue de distribution — c'est là que vivent les communes
intéressantes.

**Ce qu'on observe.** Les variables d'historique et de surface sont très
asymétriques : l'écrasante majorité des communes est à zéro, une petite minorité
concentre tout. C'est attendu, et **ce n'est pas un problème pour un modèle à
base d'arbres** (Random Forest), qui découpe par seuils et se moque de la forme
de la distribution. Ça en serait un pour une régression linéaire, qui
demanderait un passage au logarithme.

In [ ]:
asymetrie = (features[numeriques]
             .replace([np.inf, -np.inf], np.nan)
             .skew()
             .sort_values(ascending=False)
             .round(2)
             .rename("asymetrie"))
print("Asymetrie (skewness) - au-dela de 3, la distribution est tres desequilibree\n")
asymetrie.to_frame()

## 4 · Le déséquilibre de la cible

C'est **le** chiffre qui conditionne tout le reste : le choix des métriques, le
réglage du modèle, la lecture des résultats.

In [ ]:
entrainable = features[features["annee"].between(2011, jm.ANNEE_TEST)]
taux = entrainable["cible_incendie_suivant"].mean()

print(f"Lignes utilisables      : {len(entrainable):,}")
print(f"Communes qui brulent    : {entrainable['cible_incendie_suivant'].sum():,}")
print(f"Taux de positifs        : {taux:.2%}")
print(f"Ratio negatifs/positifs : 1 pour {(1 - taux) / taux:.0f}")

print("\n--- Le piege de l'accuracy ---")
print(f"Un modele qui repond TOUJOURS 'pas de feu' aurait : {1 - taux:.2%} d'accuracy")
print("...sans rien avoir appris. C'est pourquoi on ne publie PAS l'accuracy.")

In [ ]:
par_annee = features.groupby("annee")["cible_incendie_suivant"].agg(["mean", "sum"])
fig, ax = plt.subplots()
ax.bar(par_annee.index, par_annee["mean"] * 100, color="#C44E52")
ax.set_title("Part des communes ayant un feu l'annee suivante")
ax.set_ylabel("% de communes")
ax.axhline(taux * 100, color="black", ls="--", lw=1,
           label=f"moyenne 2011-{jm.ANNEE_TEST} : {taux:.2%}")
ax.legend()
plt.show()

**Ce que ce graphique dit.** Le taux varie d'une année à l'autre — les étés
exceptionnels se voient. Cette variation est une **information réelle**, mais
elle est aussi une **difficulté** : le modèle apprend un taux moyen, et se
trompera mécaniquement les années hors norme.

**À documenter comme limite :** les années extrêmes (2003, 2022) sont
précisément celles que le modèle ratera le plus.

## 5 · Corrélations entre variables — chasse aux redondances

Deux variables très corrélées apportent **une seule** information. Les garder
toutes les deux ne rend pas le modèle meilleur ; ça rend l'interprétation plus
confuse, parce que l'importance se partage entre elles au lieu de se concentrer.

In [ ]:
matrice = (entrainable[numeriques]
           .replace([np.inf, -np.inf], np.nan)
           .fillna(0)
           .corr())

fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(matrice, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeriques)))
ax.set_xticklabels(numeriques, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeriques)))
ax.set_yticklabels(numeriques, fontsize=8)
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8, label="correlation")
ax.set_title("Matrice de correlation des variables")
plt.tight_layout()
plt.show()

In [ ]:
SEUIL_REDONDANCE = 0.95

paires = []
for i, a in enumerate(numeriques):
    for b in numeriques[i + 1:]:
        r = matrice.loc[a, b]
        if abs(r) >= SEUIL_REDONDANCE:
            paires.append({"variable_1": a, "variable_2": b, "correlation": round(r, 3)})

if paires:
    print(f"Paires redondantes (|r| >= {SEUIL_REDONDANCE}) :")
    display(pd.DataFrame(paires).sort_values("correlation", key=abs, ascending=False))
    print("\n=> Envisager de n'en garder qu'une par paire.")
else:
    print(f"OK : aucune paire au-dessus de {SEUIL_REDONDANCE}.")

print("\n--- Les 10 plus fortes correlations, redondantes ou non ---")
sans_diagonale = matrice.where(~np.eye(len(matrice), dtype=bool))
top = (sans_diagonale.abs().unstack().dropna()
       .sort_values(ascending=False).head(20)[::2].round(3))
top.to_frame("correlation")

**Comment décider.** Une corrélation de 0,95 entre `nb_feux_5a` et `nb_feux_10a`
est **normale et acceptable** : les deux mesurent l'historique, mais sur des
fenêtres différentes, et l'écart entre elles porte une information (une commune
active récemment vs une commune active autrefois).

Une corrélation de 0,99 entre deux variables censées mesurer des choses
**différentes** serait au contraire le signe d'une erreur de calcul.

**On ne supprime donc pas mécaniquement au-dessus d'un seuil : on regarde si les
deux variables ont une raison d'être différentes.**

## 6 · Corrélations avec la cible — **la chasse à la fuite**

C'est la section la plus importante du notebook.

**Le raisonnement.** Notre problème est difficile : prédire un événement rare à
un an d'échéance, sans météo. Aucune variable ne devrait, **à elle seule**,
prédire très bien. Si l'une le fait, l'explication la plus probable n'est pas
qu'on a trouvé la variable miracle — c'est qu'elle **contient un morceau de la
réponse**.

In [ ]:
correlations_cible = (entrainable[numeriques + ["cible_incendie_suivant"]]
                      .replace([np.inf, -np.inf], np.nan).fillna(0)
                      .corr()["cible_incendie_suivant"]
                      .drop("cible_incendie_suivant")
                      .sort_values(key=abs, ascending=False)
                      .round(3))

fig, ax = plt.subplots(figsize=(9, 6))
couleurs = ["#C44E52" if abs(v) > 0.7 else "#4C72B0" for v in correlations_cible]
ax.barh(correlations_cible.index[::-1], correlations_cible.values[::-1],
        color=couleurs[::-1])
ax.axvline(0.7, color="crimson", ls="--", lw=1)
ax.axvline(-0.7, color="crimson", ls="--", lw=1)
ax.set_title("Correlation de chaque variable avec la cible\n"
             "(au-dela de +/-0,7 : suspect)")
plt.tight_layout()
plt.show()

correlations_cible.to_frame("correlation_avec_la_cible")

### Le test décisif : le pouvoir prédictif de **chaque variable seule**

La corrélation ne capte que les liens **linéaires**. Une fuite peut se cacher
dans un lien non linéaire et passer inaperçue.

Le test plus sûr : mesurer le **ROC-AUC de chaque variable prise isolément**.
On classe les communes selon cette seule variable et on regarde si le classement
retrouve la cible.

| AUC univarié | Interprétation |
|---|---|
| ~0,50 | la variable ne sert à rien |
| 0,60 – 0,80 | variable utile, comportement normal |
| **> 0,90** | **suspect : une variable seule ne devrait pas faire ça** |

In [ ]:
from sklearn.metrics import roc_auc_score

cible = entrainable["cible_incendie_suivant"]
resultats = []
for colonne in numeriques:
    valeurs = entrainable[colonne].replace([np.inf, -np.inf], np.nan).fillna(0)
    if valeurs.nunique() < 2:
        continue
    auc = roc_auc_score(cible, valeurs)
    resultats.append({
        "variable": colonne,
        "auc_univarie": round(max(auc, 1 - auc), 3),   # symetrise : 0,2 vaut 0,8
        "sens": "+" if auc >= 0.5 else "-",
    })

pouvoir = pd.DataFrame(resultats).sort_values("auc_univarie", ascending=False)
suspectes = pouvoir[pouvoir["auc_univarie"] > 0.90]

if len(suspectes):
    print("ALERTE - variables au pouvoir predictif anormal, a inspecter :")
    display(suspectes)
else:
    print("OK : aucune variable ne depasse 0,90 d'AUC univarie.")
    print("Aucun signe de fuite de donnees.\n")

pouvoir.reset_index(drop=True)

### Le contrôle direct : le décalage temporel est-il vraiment appliqué ?

Les deux tests précédents sont des **indices**. Voici la **preuve**.

On prend une commune au hasard parmi les plus actives, et on vérifie à la main
que son `nb_feux_5a` pour l'année A vaut bien le nombre de feux observés entre
A−5 et A−1 — et **pas** entre A−4 et A.

In [ ]:
# une commune bien fournie, pour que le test soit parlant
candidate = (incendies.groupby("code_insee").size()
             .sort_values(ascending=False).index[0])
annee_testee = 2018

feux_par_annee = (incendies[incendies["code_insee"] == candidate]
                  .groupby("annee").size())

attendu = feux_par_annee.reindex(range(annee_testee - 5, annee_testee)).fillna(0).sum()
obtenu = features.loc[
    (features["code_insee"] == candidate) & (features["annee"] == annee_testee),
    "nb_feux_5a"].iloc[0]
inclurait_annee_courante = feux_par_annee.reindex(
    range(annee_testee - 4, annee_testee + 1)).fillna(0).sum()

print(f"Commune testee : {candidate}   Annee testee : {annee_testee}\n")
print(f"Feux {annee_testee-5} a {annee_testee-1} (attendu, SANS l'annee courante) : {attendu:.0f}")
print(f"Valeur de nb_feux_5a dans le panneau                     : {obtenu:.0f}")
print(f"Ce qu'on aurait AVEC l'annee courante (= fuite)          : {inclurait_annee_courante:.0f}")
print()
if abs(obtenu - attendu) < 0.5:
    print("OK - le decalage shift(1) est bien applique : AUCUNE FUITE.")
elif abs(obtenu - inclurait_annee_courante) < 0.5:
    print("ECHEC - la fenetre inclut l'annee courante : IL Y A UNE FUITE.")
else:
    print("A verifier - la valeur ne correspond a aucun des deux cas.")

> **Ce contrôle est à montrer en soutenance.** C'est la preuve chiffrée, sur un
> cas précis, que la règle anti-fuite est appliquée. Une affirmation vaut moins
> qu'une vérification.

## 7 · Les coudes — combien de groupes, et quel rayon

Demande explicite du prof. Deux questions, deux outils.

### 7.1 · KMeans : combien de familles de communes ?

Si on écrit `k = 4` sans justification, la première question du jury sera
« pourquoi 4 ? ». Deux courbes répondent.

**L'inertie** mesure à quel point les groupes sont serrés. Elle **diminue
toujours** quand `k` augmente (avec autant de groupes que de points, elle vaut
zéro) — on ne cherche donc **pas** son minimum, mais le moment où elle **arrête
de beaucoup diminuer**.

**La silhouette** va de −1 à 1 : proche de 1, les groupes sont nets ; proche de
0, ils se chevauchent.

In [ ]:
inerties = pd.Series(infos_clusters["kmeans_inerties"]).astype(float)
inerties.index = inerties.index.astype(int)
silhouettes = pd.Series(infos_clusters["kmeans_silhouettes"]).astype(float)
silhouettes.index = silhouettes.index.astype(int)
k_retenu = infos_clusters["kmeans_groupes"]

fig, (g, d) = plt.subplots(1, 2, figsize=(13, 4.5))

g.plot(inerties.index, inerties.values, marker="o", color="#4C72B0")
g.axvline(k_retenu, color="crimson", ls="--", lw=1)
g.set_title("Methode du coude - inertie")
g.set_xlabel("nombre de groupes (k)"); g.set_ylabel("inertie")

d.plot(silhouettes.index, silhouettes.values, marker="o", color="#55A868")
d.axvline(k_retenu, color="crimson", ls="--", lw=1)
d.set_title("Score de silhouette")
d.set_xlabel("nombre de groupes (k)"); d.set_ylabel("silhouette")

fig.suptitle(f"Choix du nombre de groupes - retenu : k = {k_retenu}", y=1.02)
plt.tight_layout(); plt.show()

pd.DataFrame({"inertie": inerties.round(0), "silhouette": silhouettes.round(4)})

**Comment lire ces deux courbes ensemble.** L'inertie décroît régulièrement sans
coude franc — elle ne tranche pas. C'est la silhouette qui décide : elle reste
haute puis **s'effondre**, et c'est cette chute qui marque la limite. Au-delà,
on ne sépare plus des groupes réels, on découpe du bruit.

Le `k` retenu est celui qui **maximise la silhouette**.

### 7.2 · DBSCAN : quel rayon (`eps`) ?

DBSCAN ne demande pas un nombre de groupes mais un **rayon de voisinage**.
L'équivalent du coude s'appelle le **graphe des k-distances** : on trie tous les
points par la distance à leur k-ième plus proche voisin, et on trace la courbe.
**Le coude donne le bon `eps`.**

En dessous du coude : les points ont des voisins proches, ils sont en zone
dense. Au-dessus : ce sont des points isolés.

> Ici DBSCAN travaille sur les **coordonnées des incendies**, pas des communes.
> C'est essentiel : regrouper les 34 900 communes par proximité décrirait la
> forme de la France, pas les zones où ça brûle.

In [ ]:
from sklearn.neighbors import NearestNeighbors

MIN_SAMPLES = 8          # la meme valeur que dans jour4_model.py
RAYON_TERRE_KM = 6371

feux_geo = (incendies.dropna(subset=["latitude", "longitude"])
            [["latitude", "longitude"]].drop_duplicates())
echantillon = feux_geo.sample(min(20000, len(feux_geo)), random_state=42)
radians = np.radians(echantillon.to_numpy())

voisins = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric="haversine").fit(radians)
distances, _ = voisins.kneighbors(radians)
k_distances_km = np.sort(distances[:, -1]) * RAYON_TERRE_KM

fig, ax = plt.subplots()
ax.plot(k_distances_km, color="#4C72B0")
ax.axhline(20, color="crimson", ls="--", lw=1, label="eps retenu : 20 km")
ax.set_ylim(0, min(80, k_distances_km.max()))
ax.set_xlabel("points tries par distance croissante")
ax.set_ylabel(f"distance au {MIN_SAMPLES}e voisin (km)")
ax.set_title("Graphe des k-distances - le coude donne le bon eps")
ax.legend()
plt.show()

print(f"Points utilises : {len(echantillon):,}")
for q in (0.5, 0.75, 0.90, 0.95, 0.99):
    print(f"  {q:.0%} des points ont leur {MIN_SAMPLES}e voisin a moins de "
          f"{np.quantile(k_distances_km, q):6.2f} km")

**Comment choisir `eps` sur ce graphique.** On cherche le point où la courbe
part vers le haut. À gauche de ce coude, les points sont en zone dense ; à
droite, ils sont isolés. Choisir `eps` au niveau du coude, c'est dire : *« au
delà de cette distance, on n'est plus dans une zone, on est dans le vide. »*

Comparer la ligne rouge (20 km, la valeur retenue) à la position réelle du
coude : si le coude est nettement plus bas, `eps` est trop large et les zones
vont fusionner ; s'il est plus haut, `eps` est trop étroit et presque tout sera
classé en bruit.

## 8 · Ce que les clusters ont trouvé

Un cluster qu'on ne sait pas **nommer** est un cluster qu'on ne défendra pas en
soutenance. On regarde donc la taille de chaque groupe et son profil moyen, puis
on lui donne un nom en français.

In [ ]:
reference = features[features["annee"] == jm.ANNEE_TEST - 1]

print("--- KMeans : profils de communes ---")
profils = (reference.groupby("cluster_risque")
           .agg(communes=("code_insee", "size"),
                feux_5a=("nb_feux_5a", "mean"),
                surface_5a=("surface_5a_ha", "mean"),
                altitude=("altitude_moy", "mean"),
                densite=("densite", "mean"),
                mois_type=("mois_reference", "median"))
           .round(2))
profils["part"] = (profils["communes"] / profils["communes"].sum() * 100).round(1)
display(profils)

print("\n--- DBSCAN : zones de feu ---")
zones = (reference.groupby("cluster_spatial")
         .agg(communes=("code_insee", "size"),
              feux_5a=("nb_feux_5a", "mean"),
              lat=("latitude", "mean"),
              lon=("longitude", "mean"))
         .round(2)
         .sort_values("communes", ascending=False))
display(zones.head(10))

hors_zone = int(zones.loc[-1, "communes"]) if -1 in zones.index else 0
print(f"\nCommunes hors zone (bruit) : {hors_zone:,} "
      f"soit {hors_zone / len(reference):.1%} des communes")
print("=> C'est un resultat, pas un echec : la majorite des communes francaises")
print("   n'est effectivement pas dans un point chaud d'incendie.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
dessin = reference.dropna(subset=["latitude", "longitude"])
dans_zone = dessin[dessin["cluster_spatial"] >= 0]
hors = dessin[dessin["cluster_spatial"] < 0]

ax.scatter(hors["longitude"], hors["latitude"], s=1, c="lightgrey",
           label="hors zone")
ax.scatter(dans_zone["longitude"], dans_zone["latitude"], s=2,
           c=dans_zone["cluster_spatial"], cmap="tab20", label="zones DBSCAN")
ax.set_title("Zones de feu identifiees par DBSCAN")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal"); ax.legend(markerscale=6, loc="lower left")
plt.show()

**Le test de la carte.** Les zones colorées doivent se concentrer sur l'arc
méditerranéen, la Corse et le sud-ouest — les régions où l'on sait que les feux
sont fréquents. Si les couleurs couvrent la France entière de façon homogène,
c'est que `eps` est trop large et que DBSCAN décrit le territoire, pas les feux.

**À nommer avant la soutenance :** chaque groupe KMeans mérite une étiquette en
français, tirée du tableau de profils ci-dessus — par exemple « communes à forte
récurrence estivale » ou « communes peu exposées ».

## 9 · Conclusions

### Ce que ce notebook a vérifié

| Contrôle | Résultat attendu |
|---|---|
| Colonnes constantes | aucune |
| Valeurs manquantes | sous 5 %, hors premières années |
| Profondeur de l'historique | monte puis se stabilise → justifie le départ en 2011 |
| Redondances | paires identifiées, décision motivée |
| Corrélation avec la cible | aucune au-dessus de 0,7 |
| **AUC univarié** | **aucun au-dessus de 0,90 → pas de fuite** |
| **Contrôle direct du `shift(1)`** | **valeur = historique strictement antérieur** |
| Coude + silhouette | `k` justifié par deux courbes |
| Graphe des k-distances | `eps` justifié |
| Profils des clusters | groupes nommables |

### Ce qui reste ouvert — à écrire dans le rapport

1. **La granularité est annuelle, pas mensuelle.** `mois_reference` est la
   signature saisonnière de la commune, pas un mois choisi par l'utilisateur.
   Une commune a donc le même score en août et en février. C'est le principal
   écart avec le sujet, qui demande p. 7 « commune **+ période** ».

2. **Pas de variable de voisinage.** Le sujet mentionne p. 10 une « densité
   d'incendies dans un rayon de 10/20/50 km ». PostGIS est disponible et
   inutilisé pour ça. C'est ce qui permettrait à une commune jamais brûlée mais
   entourée de communes qui brûlent d'obtenir un score cohérent.

3. **Pas de météo** — et c'est assumé : le sujet l'exclut explicitement p. 4 de
   cette version. C'est la **limite n° 1** et la **recommandation
   d'amélioration n° 1**.

4. **Les années exceptionnelles seront mal prédites.** Le modèle apprend un
   régime moyen ; 2003 et 2022 sortent de ce régime.

### La phrase à retenir

> Une EDA sur des données brutes cherche **ce qu'elles contiennent**.
> Une EDA sur des variables construites cherche **ce qu'on a cassé en les
> construisant**. La deuxième est moins spectaculaire, et c'est elle qui évite
> de livrer un modèle qui triche sans le savoir.